In [1]:
"""
================================================================================
policy_evaluation.ipynb
================================================================================
Project     : Travel & Hospitality — Dynamic Pricing using RL
              Infotact Technical Internship — Project 2
Author      : Subhashree Behera
Week        : Week 4 — Policy Evaluation & Business Dashboard
Role        : Member 1 — Policy Evaluation

--------------------------------------------------------------------------------
Objective:
    Evaluate the trained DQN agent against baseline pricing strategies.
    Compare DQN with Random, Fixed High (₹250), and Fixed Low (₹50)
    policies across 100 simulated booking seasons.

    This notebook answers the core business question:
    "Did our DQN agent learn a genuinely better pricing strategy
    than simple rule-based approaches?"

--------------------------------------------------------------------------------
Metrics Calculated:
    - Average episode reward (total revenue per season)
    - Cumulative reward across all evaluation episodes
    - Revenue improvement % over each baseline
    - Rooms sold, rooms unsold (spoilage) per policy

--------------------------------------------------------------------------------
Deliverables:
    - Policy comparison table (DQN vs 3 baselines)
    - Cumulative reward curve
    - Average reward bar chart
    - Revenue improvement summary
================================================================================
"""

'\n================================================================================\npolicy_evaluation.ipynb\n================================================================================\nProject     : Travel & Hospitality — Dynamic Pricing using RL\n              Infotact Technical Internship — Project 2\nAuthor      : Subhashree Behera\nWeek        : Week 4 — Policy Evaluation & Business Dashboard\nRole        : Member 1 — Policy Evaluation\n\n--------------------------------------------------------------------------------\nObjective:\n    Evaluate the trained DQN agent against baseline pricing strategies.\n    Compare DQN with Random, Fixed High (₹250), and Fixed Low (₹50)\n    policies across 100 simulated booking seasons.\n\n    This notebook answers the core business question:\n    "Did our DQN agent learn a genuinely better pricing strategy\n    than simple rule-based approaches?"\n\n--------------------------------------------------------------------------------\nMetrics Ca

In [2]:
# Cell 2: Imports and Constants
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ── Constants (must match mdp_design.py exactly) ──────────
PRICE_LEVELS     = [50, 100, 150, 200, 250]
N_ACTIONS        = len(PRICE_LEVELS)
TOTAL_INVENTORY  = 100
TOTAL_DAYS       = 30
RANDOM_STATE     = 42
N_EVAL_EPISODES  = 100

# Output folder
Path('plots').mkdir(exist_ok=True)

# Plot style
plt.style.use('seaborn-v0_8-darkgrid')

print("Setup complete:")
print(f"  Price levels    : {PRICE_LEVELS}")
print(f"  Eval episodes   : {N_EVAL_EPISODES}")
print(f"  Total inventory : {TOTAL_INVENTORY} rooms")
print(f"  Season length   : {TOTAL_DAYS} days")

Setup complete:
  Price levels    : [50, 100, 150, 200, 250]
  Eval episodes   : 100
  Total inventory : 100 rooms
  Season length   : 30 days


In [3]:
# Cell 3: Demand Function and DQN Agent
# ── Demand Function ────────────────────────────────────────
def demand_function(price, days_remaining):
    """
    Stochastic customer booking demand.
    Higher price → fewer bookings.
    Fewer days → urgency increases bookings.
    Formula from Week 1 mdp_design.py.
    """
    prob = 0.8 - 0.002 * price + 0.3 * (1 / max(days_remaining, 1))
    prob = max(0.0, min(1.0, prob))
    return int(np.random.binomial(5, prob))


# ── DQN Agent ─────────────────────────────────────────────
# Replace with: from dqn_model import DQNAgent
# when Member 1's trained model is available

class DQNAgent:
    """
    Smart pricing agent that learned from training.
    Mimics real DQN behavior: high prices early,
    drops prices as deadline approaches.
    """
    def __init__(self):
        np.random.seed(RANDOM_STATE)
        self.name = "DQN Agent"

    def select_action(self, state):
        inventory, days = state
        # Learned behavior: urgency-aware pricing
        if days > 20:
            return np.random.choice([2, 3, 4])  # premium pricing
        elif days > 10:
            return np.random.choice([1, 2, 3])  # standard pricing
        else:
            return np.random.choice([0, 1, 2])  # clear inventory

print("DQN Agent and demand function ready ✓")

DQN Agent and demand function ready ✓


In [4]:
# Cell 4: Episode Runner — works for both DQN and baselines

def run_episode(policy='dqn', agent=None):
    """
    Run one complete 30-day booking season.

    Args:
        policy : 'dqn', 'random', 'fixed_high', 'fixed_low'
        agent  : DQNAgent instance (only needed for policy='dqn')

    Returns:
        dict with total_revenue, rooms_sold, rooms_unsold,
        price_history, reward_history
    """
    inventory      = TOTAL_INVENTORY
    days           = TOTAL_DAYS
    total_revenue  = 0
    rooms_sold     = 0
    price_history  = []
    reward_history = []

    while days > 0 and inventory > 0:
        # Select price based on policy
        if policy == 'dqn' and agent:
            action_idx = agent.select_action([inventory, days])
            price      = PRICE_LEVELS[action_idx]
        elif policy == 'fixed_high':
            price = 250
        elif policy == 'fixed_low':
            price = 50
        else:  # random
            price = np.random.choice(PRICE_LEVELS)

        # Simulate demand
        bookings      = min(demand_function(price, days), inventory)
        reward        = price * bookings

        # Update state
        total_revenue  += reward
        rooms_sold     += bookings
        inventory      -= bookings
        days           -= 1
        price_history.append(price)
        reward_history.append(reward)

    return {
        'total_revenue' : total_revenue,
        'rooms_sold'    : rooms_sold,
        'rooms_unsold'  : inventory,
        'price_history' : price_history,
        'reward_history': reward_history
    }


# Quick test
np.random.seed(42)
agent  = DQNAgent()
result = run_episode('dqn', agent)
print(f"Test episode result:")
print(f"  Total Revenue : ₹{result['total_revenue']:.0f}")
print(f"  Rooms Sold    : {result['rooms_sold']}")
print(f"  Rooms Unsold  : {result['rooms_unsold']}")

Test episode result:
  Total Revenue : ₹11100
  Rooms Sold    : 81
  Rooms Unsold  : 19


In [5]:
# Cell 5: Run 100 evaluation episodes for all policies

np.random.seed(RANDOM_STATE)
agent   = DQNAgent()
results = {}

policies = ['dqn', 'random', 'fixed_high', 'fixed_low']

print("=" * 55)
print(f"RUNNING {N_EVAL_EPISODES} EPISODES PER POLICY")
print("=" * 55)

for policy in policies:
    episodes = []
    for _ in range(N_EVAL_EPISODES):
        ep = run_episode(policy, agent if policy=='dqn' else None)
        episodes.append(ep)
    results[policy] = episodes

    revenues = [ep['total_revenue'] for ep in episodes]
    print(f"{policy:<12}: Mean ₹{np.mean(revenues):.2f} "
          f"| Std ₹{np.std(revenues):.2f}")

print("=" * 55)
print("All episodes complete ✓")

RUNNING 100 EPISODES PER POLICY
dqn         : Mean ₹10934.50 | Std ₹889.71
random      : Mean ₹10623.00 | Std ₹975.51
fixed_high  : Mean ₹12670.00 | Std ₹1357.33
fixed_low   : Mean ₹4996.50 | Std ₹26.70
All episodes complete ✓
